<a href="https://colab.research.google.com/github/phillip-jaeslee/PULSIM/blob/v_torch/PULSIM_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# PULSIM — shaped pulse simulator

Simulate the excitation profile of shaped RF pulses and multi-pulse sequences,
entirely in the browser. No installation, no licence, no local compute.

**How to use:** `Runtime` -> `Run all`, then edit the fields in the forms below and
re-run the cell you changed.

| Section | What it does |
|---|---|
| 1. Setup | Clones PULSIM and installs dependencies |
| 2. Single pulse | Pick one shape, see its excitation profile |
| 3. Pulse sequence | Chain several pulses, see the combined profile |
| 4. Scripting | The same simulation written directly in code |
| 5. Shape catalogue | Every shape available, with its registry name |


## 1. Setup

Run this once per session. Takes about 30 seconds.


In [ ]:
#@title Install PULSIM
!git clone --quiet --branch v_torch https://github.com/phillip-jaeslee/PULSIM
%cd PULSIM
# requirements.txt lists `ffmpeg`, which on PyPI is an abandoned stub, and
# `inflect`/`ipympl`, which this notebook does not need. Install only what is used.
!pip install --quiet scipy "numpy>=1.25" matplotlib joblib
print("PULSIM ready.")


In [ ]:
#@title Load the simulator  { display-mode: "form" }
import time

import numpy as np
import matplotlib.pyplot as plt

from rf_shape import RFShape
from pulse_oo import Pulse
from pulse_sequence import PulseSequence
from backend import NumpyBackend

# Gyromagnetic ratios, MHz/T. Defined here rather than imported from
# input_parameter.py, which pulls in `inflect` and contains interactive
# input() prompts that do not belong in a notebook.
GAMMA = {"H": 42.577478518, "D": 6.536, "T": 45.415,
         "13C": 10.7084, "15N": -4.316, "19F": 40.052, "31P": 17.235}


def gyro_ratio(nucleus):
    if nucleus not in GAMMA:
        raise ValueError(f"unknown nucleus {nucleus!r}; known: {list(GAMMA)}")
    return GAMMA[nucleus]


def make_backend(kind, Gamma):
    """Torch batches every offset into one operation per time step; NumPy loops
    over offsets in Python and is roughly 50x slower. Use NumPy as the reference
    implementation, Torch for anything interactive."""
    if kind == "torch":
        try:
            import torch  # noqa: F401
            from backend import TorchBackend
            return TorchBackend(Gamma=Gamma)
        except Exception as exc:
            print(f"torch unavailable ({exc}) - falling back to numpy")
    return NumpyBackend(Gamma=Gamma)


def run_timed(seq, M, df):
    t0 = time.time()
    M = seq.run(M, df)
    print(f"  simulated in {time.time() - t0:.2f} s")
    return M


def make_M(n_offsets, init="z", M0=1.0):
    """Starting magnetization: one vector per offset, as a (3, n) array."""
    vec = {"x": [M0, 0, 0], "y": [0, M0, 0], "z": [0, 0, M0]}
    if init not in vec:
        raise ValueError(f"init must be x, y or z; got {init!r}")
    return np.tile(np.array(vec[init], dtype=float), (n_offsets, 1)).T


def plot_result(seq, M, df_khz, title=""):
    """Top: RF amplitude and phase vs time. Bottom: excitation profile vs offset."""
    fig, axs = plt.subplots(3, 1, figsize=(7, 8), height_ratios=[1, 1, 2])

    rf = np.asarray(seq.rf)
    t = np.asarray(seq.time)
    axs[0].plot(t, np.abs(rf), color="tab:blue")
    axs[0].set(xlabel="time (ms)", ylabel="|B1| (mT)", title=title or "RF amplitude")

    axs[1].plot(t, np.asarray(seq.phase), color="tab:orange")
    axs[1].set(xlabel="time (ms)", ylabel="phase (deg)")

    df_hz = df_khz * 1000.0
    axs[2].plot(df_hz, M[0], label="Mx")
    axs[2].plot(df_hz, M[1], label="My")
    axs[2].plot(df_hz, M[2], label="Mz")
    axs[2].plot(df_hz, np.hypot(M[0], M[1]), "k--", lw=1, label="|Mxy|")
    axs[2].set(xlabel="offset (Hz)", ylabel="magnetization",
               title="Excitation profile")
    axs[2].axhline(0, color="0.8", lw=0.5)
    axs[2].legend(loc="best", fontsize=8)

    fig.tight_layout()
    plt.show()
    return fig

print("Loaded.", len(RFShape._registry), "pulse shapes available.")
try:
    import torch
    print("Backend: torch", torch.__version__,
          "(GPU)" if torch.cuda.is_available() else "(CPU)")
except Exception:
    print("Backend: numpy only. Expect ~40 s per pulse at 1000 offsets x 1000 points;"
          " reduce n_offsets to ~200 to stay interactive.")


## 2. Single pulse

Pick one shape and see what it does.

`duration` is in **ms**, `bandwidth` in **kHz**, `flip` in **degrees**.
For `shape = file`, put the path to a shape file in `file_path`
(JCAMP-DX `##XYPOINTS`, JEOL `.jhl`, and Bruker-style files are all read);
otherwise leave it blank.


In [ ]:
#@title Simulate one pulse  { display-mode: "form" }
#@markdown ### Spin
nucleus   = "H"   #@param ["H", "D", "T", "13C", "15N", "19F", "31P"]
init_vect = "z"   #@param ["x", "y", "z"]

#@markdown ### Sweep
bandwidth = 6     #@param {type:"number"}
n_offsets = 1000  #@param {type:"integer"}
engine    = "torch"  #@param ["torch", "numpy"]

#@markdown ### Pulse
shape     = "gausscasq5"  #@param ["eburp1", "eburp2", "iburp1", "iburp2", "uburp", "reburp", "gausscasg3", "gausscasg4", "gausscasq3", "gausscasq5", "hermite", "seduce1", "sneeze", "qsneeze", "esnob", "i2snob", "i3snob", "rsnob", "dsnob", "hypsec", "wurst", "smoothedchirp", "tanhtan", "cawurst", "cagauss", "calorentz", "capowhsec", "sinc", "cos", "sincos", "hard", "file"]
duration  = 3.0   #@param {type:"number"}
points    = 1000  #@param {type:"integer"}
flip      = 90.0  #@param {type:"number"}
axis      = "x"   #@param ["x", "y", "z"]
file_path = ""    #@param {type:"string"}

Gamma = gyro_ratio(nucleus)
df = np.linspace(-bandwidth / 2, bandwidth / 2, n_offsets)

kwargs = {"duration": duration, "points": points}
if shape == "file":
    kwargs = {"duration": duration, "path": file_path}

pulse = Pulse(
    RFShape.create(shape, **kwargs),
    flip=np.deg2rad(flip),
    axis=axis,
    backend=make_backend(engine, Gamma),
)

seq = PulseSequence([pulse])
M = run_timed(seq, make_M(n_offsets, init_vect), df)

plot_result(seq, M, df, title=f"{shape}  {flip:g}deg  {duration:g} ms")


## 3. Pulse sequence

Chain several pulses. Each is applied in order to the same magnetization.

Set `n_pulses` to how many you want, then fill in that many rows below.
Rows beyond `n_pulses` are ignored, so you can leave them alone.


In [ ]:
#@title Build a sequence  { display-mode: "form" }
#@markdown ### Spin and sweep
nucleus   = "H"   #@param ["H", "D", "T", "13C", "15N", "19F", "31P"]
init_vect = "z"   #@param ["x", "y", "z"]
bandwidth = 6     #@param {type:"number"}
n_offsets = 1000  #@param {type:"integer"}
engine    = "torch"  #@param ["torch", "numpy"]
n_pulses  = 3     #@param {type:"slider", min:1, max:5, step:1}

#@markdown ---
#@markdown ### Pulse 1
shape_1 = "gausscasq5"  #@param {type:"string"}
flip_1 = 90.0    #@param {type:"number"}
dur_1 = 3.0      #@param {type:"number"}
pts_1 = 1000     #@param {type:"integer"}
axis_1 = "x"     #@param ["x", "y", "z"]
path_1 = ""      #@param {type:"string"}

#@markdown ### Pulse 2
shape_2 = "hard" #@param {type:"string"}
flip_2 = 180.0   #@param {type:"number"}
dur_2 = 0.02     #@param {type:"number"}
pts_2 = 100      #@param {type:"integer"}
axis_2 = "x"     #@param ["x", "y", "z"]
path_2 = ""      #@param {type:"string"}

#@markdown ### Pulse 3
shape_3 = "gausscasq5"  #@param {type:"string"}
flip_3 = 90.0    #@param {type:"number"}
dur_3 = 3.0      #@param {type:"number"}
pts_3 = 1000     #@param {type:"integer"}
axis_3 = "x"     #@param ["x", "y", "z"]
path_3 = ""      #@param {type:"string"}

#@markdown ### Pulse 4
shape_4 = "hard" #@param {type:"string"}
flip_4 = 90.0    #@param {type:"number"}
dur_4 = 0.02     #@param {type:"number"}
pts_4 = 100      #@param {type:"integer"}
axis_4 = "x"     #@param ["x", "y", "z"]
path_4 = ""      #@param {type:"string"}

#@markdown ### Pulse 5
shape_5 = "hard" #@param {type:"string"}
flip_5 = 90.0    #@param {type:"number"}
dur_5 = 0.02     #@param {type:"number"}
pts_5 = 100      #@param {type:"integer"}
axis_5 = "x"     #@param ["x", "y", "z"]
path_5 = ""      #@param {type:"string"}

# ---- build ----------------------------------------------------------------
Gamma = gyro_ratio(nucleus)
df = np.linspace(-bandwidth / 2, bandwidth / 2, n_offsets)
backend = make_backend(engine, Gamma)

rows = [
    (shape_1, flip_1, dur_1, pts_1, axis_1, path_1),
    (shape_2, flip_2, dur_2, pts_2, axis_2, path_2),
    (shape_3, flip_3, dur_3, pts_3, axis_3, path_3),
    (shape_4, flip_4, dur_4, pts_4, axis_4, path_4),
    (shape_5, flip_5, dur_5, pts_5, axis_5, path_5),
][:n_pulses]

seq = PulseSequence()
for i, (shp, flp, dur, pts, ax, pth) in enumerate(rows, start=1):
    shp = shp.strip().lower()
    kw = {"duration": dur, "path": pth} if shp == "file" else {"duration": dur, "points": pts}
    seq.append(Pulse(RFShape.create(shp, **kw), flip=np.deg2rad(flp), axis=ax, backend=backend))
    print(f"  {i}. {shp:<14} {flp:>6.1f} deg   {dur:>7.3f} ms   axis {ax}")

M = run_timed(seq, make_M(n_offsets, init_vect), df)
plot_result(seq, M, df, title=f"sequence of {len(seq)} pulses")


## 4. The same thing, in code

The form above is convenience. Underneath, a whole sequence is this:

```python
seq = PulseSequence([
    Pulse(RFShape.create("gausscasq5", duration=3.0,  points=1000), np.pi/2),
    Pulse(RFShape.create("hard",       duration=0.02, points=100),  np.pi),
    Pulse(RFShape.create("gausscasq5", duration=3.0,  points=1000), np.pi/2),
])
M = seq.run(make_M(1000, "z"), df)
```

Which makes comparisons easy — here is every Gaussian-cascade member at once.


In [ ]:
#@title Compare several shapes
bandwidth = 6
n_offsets = 600
df = np.linspace(-bandwidth / 2, bandwidth / 2, n_offsets)
backend = make_backend("torch", gyro_ratio("H"))

to_compare = ["gausscasg3", "gausscasg4", "gausscasq3", "gausscasq5"]

fig, ax = plt.subplots(figsize=(7, 4))
for name in to_compare:
    seq = PulseSequence([
        Pulse(RFShape.create(name, duration=3.0, points=1000),
              flip=np.deg2rad(90), backend=backend)
    ])
    M = seq.run(make_M(n_offsets, "z"), df)
    ax.plot(df * 1000, np.hypot(M[0], M[1]), label=name)

ax.set(xlabel="offset (Hz)", ylabel="|Mxy|",
       title="90 deg, 3 ms — Gaussian cascade family")
ax.legend()
fig.tight_layout()
plt.show()


## 5. Shape catalogue

Every registered shape name, as accepted by `RFShape.create(...)`.


In [ ]:
#@title List every available shape
names = sorted(RFShape._registry)
print(f"{len(names)} shapes registered\n")
for i in range(0, len(names), 4):
    print("  " + "".join(f"{n:<24}" for n in names[i:i + 4]))


---

### Known limitations of this version

- **No delay element.** A `PulseSequence` chains pulses back to back; a gap between
  pulses (free precession) cannot yet be represented. Sequences with inter-pulse
  delays will therefore be slightly wrong.
- **`axis` is limited to x, y, z.** Arbitrary RF phase is not yet supported, and only
  the `x` branch has been verified numerically.
- **No relaxation during pulses.** T1 and T2 are ignored inside a pulse.
- **Speed.** The NumPy backend loops over offsets in Python: measured at ~7.5 s for
  400 offsets x 500 points, extrapolating to ~36 s per pulse at 1000 x 1000. Torch
  batches all offsets per time step and is the default here. Note that `TorchBackend`
  currently moves data between host and device on *every* time step, so it is not yet
  as fast as it could be.

### Citing

If PULSIM is useful in your work, please cite the repository:
<https://github.com/phillip-jaeslee/PULSIM>
